# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['RQQYNVVGZI', 'KJNMXOMCMW'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[18, 17, 17, 25, 14, 22, 22,  7, 26,  9],
       [11, 10, 14, 13, 24, 15, 13,  3, 13, 23]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  9, 26,  7, 22, 22, 14, 25, 17, 17],
       [ 0, 23, 13,  3, 13, 15, 24, 13, 14, 10]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 9, 26,  7, 22, 22, 14, 25, 17, 17, 18],
       [23, 13,  3, 13, 15, 24, 13, 14, 10, 11]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def build(self, input_shape):
        # 显式声明build，避免Keras提示“未实现build方法”的warning
        super(mySeq2SeqModel, self).build(input_shape)
        
        
    def call(self, enc_ids, dec_ids):
        '''
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # 1) 编码器：把输入token映射到向量，并得到每个时间步输出与最终状态
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out:[b, src_len, h], enc_state:[b, h]
        
        # 2) 解码器输入做embedding（训练时是teacher forcing输入）
        dec_emb = self.embed_layer(dec_ids)  # [b, tgt_len, emb]
        
        # 3) 准备解码初始状态：state[0]作为attention查询向量，state[1]作为RNN隐状态
        attn_query = enc_out[:, -1, :]
        dec_state = enc_state
        
        # 4) 先对encoder输出做线性变换，供双线性attention打分使用
        proj_enc = self.dense_attn(enc_out)  # [b, src_len, h]
        
        # 5) 按时间步解码，每一步都重新计算attention上下文
        logits_collect = []
        for dec_t in tf.unstack(dec_emb, axis=1):
            # 5.1 用当前query与所有encoder位置做点积打分
            score = tf.reduce_sum(proj_enc * tf.expand_dims(attn_query, axis=1), axis=-1)  # [b, src_len]
            # 5.2 softmax得到注意力权重
            attn_w = tf.nn.softmax(score, axis=-1)  # [b, src_len]
            # 5.3 加权求和得到上下文向量
            context = tf.reduce_sum(enc_out * tf.expand_dims(attn_w, axis=-1), axis=1)  # [b, h]
            
            # 5.4 将当前decoder输入和context拼接后送入RNNCell
            rnn_inp = tf.concat([dec_t, context], axis=-1)  # [b, emb+h]
            h_t, [dec_state] = self.decoder_cell(rnn_inp, [dec_state])
            
            # 5.5 当前隐藏状态映射到词表logits
            step_logits = self.dense(h_t)  # [b, v_sz]
            logits_collect.append(step_logits)
            
            # 5.6 更新下一步attention查询向量
            attn_query = h_t
        
        # 6) 还原成三维logits，供损失函数逐时间步计算
        logits = tf.stack(logits_collect, axis=1)  # [b, tgt_len, v_sz]
        return logits
    
    
    def encode(self, enc_ids):
        # 编码并返回：encoder各时刻输出 + 解码初始状态（query, rnn_state）
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
    
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        # 1) 取出当前attention查询向量与decoder隐状态
        attn_query, dec_state = state
        
        # 2) 计算双线性attention权重
        proj_enc = self.dense_attn(enc_out)  # [b, src_len, h]
        score = tf.reduce_sum(proj_enc * tf.expand_dims(attn_query, axis=1), axis=-1)  # [b, src_len]
        attn_w = tf.nn.softmax(score, axis=-1)
        context = tf.reduce_sum(enc_out * tf.expand_dims(attn_w, axis=-1), axis=1)  # [b, h]
        
        # 3) 当前输入token做embedding，并和context拼接后做一步解码
        inp_emb = self.embed_layer(x)  # [b, emb]
        rnn_inp = tf.concat([inp_emb, context], axis=-1)  # [b, emb+h]
        h_t, [dec_state] = self.decoder_cell(rnn_inp, [dec_state])
        
        # 4) 映射到词表并贪心选取概率最大的token
        logits = self.dense(h_t)  # [b, v_sz]
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        
        # 5) 返回预测token以及下一时刻状态（新的query和rnn_state）
        state = [h_t, dec_state]
        return out, state

# Loss函数以及训练逻辑

In [7]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(5000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [8]:
optimizer = optimizers.Adam(0.0003)
model = mySeq2SeqModel()
# 先做一次dummy前向，让子层权重按真实路径创建，避免build相关warning
_ = model(tf.zeros([1, 20], dtype=tf.int32), tf.zeros([1, 20], dtype=tf.int32))
train(model, optimizer, seqlen=20)

step 0 : loss 3.3048034
step 500 : loss 0.66979325
step 1000 : loss 0.30681732
step 1500 : loss 0.1217304
step 2000 : loss 0.39268512
step 2500 : loss 0.080160104
step 3000 : loss 0.18906483
step 3500 : loss 0.21787766
step 4000 : loss 0.23765287
step 4500 : loss 0.1097525


<tf.Tensor: shape=(), dtype=float32, numpy=0.03176989033818245>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [9]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('NKQIJZJQHLJIIWMFMSUK', 'KUSMFMWIIJLHQJZJIQKN'), ('FWWARQIWUHKXYLDNPBDZ', 'ZDBPNDLYXKHUWIQRAWWF'), ('MLLZJVDTKRPNFSAZQWFH', 'HFWQZASFNPRKTDVJZLLM'), ('IUUXMVALCBEBBIOSNKDB', 'BDKNSOIBBEBCLAVMXUUI'), ('UMHRIEIZMJUCSKRMGGNY', 'YNGGMRKSCUJMZIEIRHMU'), ('GDSAXYXJCOKABNHXORSR', 'RSROXHNBAKOCJXYXASDG'), ('GATNNUPJALSOMWILTWJR', 'RJWTLIWMOSLAJPUNNTAG'), ('ZBYZWFSQNESHCLFUATNB', 'BNTAUFLCHSENQSFWZYBZ'), ('DQLFCBWSATRGHVBUJQJT', 'TJQJUBVHGRTASWBCFLQD'), ('PNLVOJKMDFCVZKKVPEBC', 'CBEPVKKZVCFDMKJOVLNP'), ('PYUHATOEIWLZRYOFNDES', 'SEDNFOYRZLWIEOTAHUYP'), ('CHQGPUSVMGPEHSIJUHNV', 'VNHUJISHEPGMVSUPGQHC'), ('ZYMUVJPPCBFUNWKTBCMA', 'AMCBTKWNUFBCPPJVUMYZ'), ('LATEJXLKJXRAZCYLUQWH', 'HWQULYCZARXJKLXJETAC'), ('GCFVIGYGJHBURNTBDGBH', 'HBGDBTNRUBHJGYGIVFCG'), ('CAQNQIODURXDSXTDMSRG', 'GRSMDTXSDXRUDOIQNQAC'), ('PGV